# Netflix Prize Dataset — Exploratory Data Analysis

This notebook performs a comprehensive exploratory data analysis of the Netflix Prize dataset.
We investigate rating distributions, user activity, movie popularity, sparsity, and temporal trends.

**Author:** Netflix Prize Recommendation System Project  
**Dataset:** Netflix Prize (100M+ ratings, 480K users, 17K movies)  
**Goal:** Understand data characteristics before modelling.

In [ ]:
# ── Setup & Imports ───────────────────────────────────────────────────────────
import sys
import logging
from pathlib import Path

# Make the project src package importable from any working directory
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

try:
    import polars as pl
    HAS_POLARS = True
except ImportError:
    HAS_POLARS = False
    print('polars not installed — using pandas only')

%matplotlib inline
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
})
sns.set_palette('muted')

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
log = logging.getLogger(__name__)

DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR  = DATA_DIR / 'raw'
PROC_DIR = DATA_DIR / 'processed'
FIG_DIR  = PROJECT_ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Data dir     : {DATA_DIR}')
print(f'Polars avail : {HAS_POLARS}')

## 1. Data Loading

We use the project's `NetflixDataLoader` utility to load the pre-processed parquet files.
If the processed files are absent the loader falls back to raw `.txt` files and
creates the parquet cache automatically.

In [ ]:
from src.data.loader import NetflixDataLoader

loader = NetflixDataLoader(data_dir=str(DATA_DIR))
df = loader.load_ratings()   # returns a pandas DataFrame

print('Shape      :', df.shape)
print('Columns    :', df.columns.tolist())
print()
print('dtypes:')
print(df.dtypes)
print()
print('Memory usage: {:.1f} MB'.format(df.memory_usage(deep=True).sum() / 1e6))
df.head(10)

In [ ]:
# Quick sanity checks
n_ratings = len(df)
n_users   = df['user_id'].nunique()
n_movies  = df['movie_id'].nunique()
date_min  = df['date'].min() if 'date' in df.columns else 'N/A'
date_max  = df['date'].max() if 'date' in df.columns else 'N/A'

print(f'Total ratings : {n_ratings:,}')
print(f'Unique users  : {n_users:,}')
print(f'Unique movies : {n_movies:,}')
print(f'Date range    : {date_min} to {date_max}')

## 2. Rating Distribution

Netflix ratings are integer-valued in **[1, 5]**.  
Understanding the distribution tells us whether the dataset is skewed toward higher ratings
(a common pattern in voluntary feedback), which will influence loss function choices.

In [ ]:
# Descriptive statistics
print('=== Rating Descriptive Statistics ===')
print(df['rating'].describe().round(4))
print()
print('=== Value Counts ===')
vc = df['rating'].value_counts().sort_index()
for r, cnt in vc.items():
    pct = cnt / n_ratings * 100
    bar = '#' * int(pct)
    print(f'  {r} stars  {cnt:>12,}  ({pct:5.2f}%)  {bar}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
ax = axes[0]
rating_counts = df['rating'].value_counts().sort_index()
bars = ax.bar(rating_counts.index, rating_counts.values,
              color=sns.color_palette('muted', 5), edgecolor='white', linewidth=0.8)
ax.set_xlabel('Star Rating')
ax.set_ylabel('Number of Ratings')
ax.set_title('Rating Frequency Distribution')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
for bar, val in zip(bars, rating_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5e5,
            f'{val/1e6:.1f}M', ha='center', va='bottom', fontsize=10)

# Pie chart
ax2 = axes[1]
ax2.pie(rating_counts.values, labels=[f'{r} star' for r in rating_counts.index],
        autopct='%1.1f%%', startangle=140,
        colors=sns.color_palette('muted', 5))
ax2.set_title('Rating Share (%)')

plt.tight_layout()
plt.savefig(FIG_DIR / 'rating_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. User Activity Analysis

The Netflix dataset is known to have a **heavy-tailed** user-activity distribution:
a small fraction of power users contribute disproportionately many ratings.
We study this via per-user rating counts and their distribution.

In [ ]:
user_activity = df.groupby('user_id')['rating'].count().rename('n_ratings')

print('=== Ratings per User — Statistics ===')
print(user_activity.describe().round(2))
print()
print('=== Percentiles ===')
for p in [50, 75, 90, 95, 99]:
    val = np.percentile(user_activity, p)
    print(f'  p{p:>5.1f}: {val:>8,.0f} ratings')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Log-scale histogram of ratings per user
ax = axes[0]
ax.hist(user_activity, bins=100, color='steelblue', edgecolor='white', linewidth=0.5)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Ratings per User (log scale)')
ax.set_ylabel('Number of Users (log scale)')
ax.set_title('User Activity Distribution (log-log)')

# Cumulative distribution
ax2 = axes[1]
sorted_activity = np.sort(user_activity.values)[::-1]
cum_pct = np.cumsum(sorted_activity) / sorted_activity.sum() * 100
user_pct = np.arange(1, len(sorted_activity)+1) / len(sorted_activity) * 100
ax2.plot(user_pct, cum_pct, color='coral', linewidth=2)
ax2.axhline(80, color='gray', linestyle='--', linewidth=1, label='80% of ratings')
ax2.set_xlabel('Top X% of Users')
ax2.set_ylabel('Cumulative % of Ratings')
ax2.set_title('Cumulative Rating Contribution by Users')
ax2.legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'user_activity.png', dpi=150, bbox_inches='tight')
plt.show()

idx_80 = np.searchsorted(cum_pct, 80)
print(f'Top {user_pct[idx_80]:.1f}% of users contribute 80% of all ratings.')

## 4. Movie Popularity Analysis

Similar to users, movies exhibit a **long-tail popularity distribution**.
The top-20 most-rated movies dominate the rating counts, while the majority of the
17,770-movie catalog receives very few ratings — a key challenge for collaborative filtering.

In [ ]:
movie_popularity = df.groupby('movie_id')['rating'].agg(['count', 'mean']).rename(
    columns={'count': 'n_ratings', 'mean': 'avg_rating'}
)

print('=== Movie Popularity Statistics ===')
print(movie_popularity.describe().round(2))

titles_path = RAW_DIR / 'movie_titles.csv'
if titles_path.exists():
    titles_df = pd.read_csv(titles_path, header=None,
                            names=['movie_id', 'year', 'title'],
                            encoding='latin-1')
    movie_popularity = movie_popularity.reset_index().merge(
        titles_df[['movie_id','title']], on='movie_id', how='left'
    ).set_index('movie_id')
    label_col = 'title'
else:
    movie_popularity['title'] = 'Movie ' + movie_popularity.index.astype(str)
    label_col = 'title'

top20 = movie_popularity.nlargest(20, 'n_ratings')
print('\nTop-20 most rated movies:')
print(top20[['n_ratings', 'avg_rating', 'title']].to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top-20 bar chart
ax = axes[0]
labels = [str(t)[:35] for t in top20['title'].values]
ax.barh(range(20), top20['n_ratings'].values[::-1],
        color=sns.color_palette('coolwarm_r', 20))
ax.set_yticks(range(20))
ax.set_yticklabels(labels[::-1], fontsize=9)
ax.set_xlabel('Number of Ratings')
ax.set_title('Top-20 Most-Rated Movies')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))

# Long-tail: histogram of ratings per movie
ax2 = axes[1]
ax2.hist(movie_popularity['n_ratings'], bins=80, color='teal', edgecolor='white', linewidth=0.4)
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel('Ratings per Movie (log scale)')
ax2.set_ylabel('Number of Movies (log scale)')
ax2.set_title('Movie Popularity Long-Tail Distribution')

plt.tight_layout()
plt.savefig(FIG_DIR / 'movie_popularity.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Sparsity Analysis

Sparsity is the fraction of (user, movie) pairs that have **no observed rating**.
Most real-world recommendation datasets are extremely sparse (>99%),
which motivates the use of collaborative filtering techniques.

In [ ]:
possible_pairs = n_users * n_movies
sparsity = 1.0 - n_ratings / possible_pairs

print(f'Users          : {n_users:>12,}')
print(f'Movies         : {n_movies:>12,}')
print(f'Observed pairs : {n_ratings:>12,}')
print(f'Possible pairs : {possible_pairs:>12,}')
print(f'Sparsity       : {sparsity:.6f}  ({sparsity*100:.4f}%)')
print(f'Density        : {1-sparsity:.6f}  ({(1-sparsity)*100:.4f}%)')

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(['Observed\nRatings', 'Missing\nPairs'],
       [n_ratings / possible_pairs * 100, sparsity * 100],
       color=['steelblue', 'lightcoral'], edgecolor='white')
ax.set_ylabel('Percentage of All Pairs (%)')
ax.set_title('Matrix Density vs. Sparsity')
for i, v in enumerate([n_ratings/possible_pairs*100, sparsity*100]):
    ax.text(i, v + 0.5, f'{v:.3f}%', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'sparsity.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Temporal Analysis

Rating volume changed substantially over the Netflix Prize contest period (2000-2005).
Temporal patterns can reveal data collection artifacts and may inform time-aware recommendation strategies.

In [ ]:
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['year_month'] = df['date'].dt.to_period('M')

    monthly = df.groupby('year_month').agg(
        n_ratings=('rating', 'count'),
        avg_rating=('rating', 'mean')
    ).reset_index()
    monthly['year_month_dt'] = monthly['year_month'].dt.to_timestamp()

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    axes[0].fill_between(monthly['year_month_dt'], monthly['n_ratings'], alpha=0.6, color='steelblue')
    axes[0].plot(monthly['year_month_dt'], monthly['n_ratings'], color='steelblue', linewidth=1.5)
    axes[0].set_ylabel('Monthly Ratings')
    axes[0].set_title('Rating Volume Over Time')
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

    axes[1].plot(monthly['year_month_dt'], monthly['avg_rating'],
                 color='coral', linewidth=2, marker='o', markersize=3)
    axes[1].axhline(df['rating'].mean(), color='gray', linestyle='--',
                    label=f'Overall mean = {df["rating"].mean():.3f}')
    axes[1].set_ylabel('Average Rating')
    axes[1].set_title('Average Rating Over Time')
    axes[1].legend()
    axes[1].set_xlabel('Date')

    plt.tight_layout()
    plt.savefig(FIG_DIR / 'temporal_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Date column not available; skipping temporal analysis.')

## 7. Key Insights

| Metric | Value | Implication |
|---|---|---|
| Total ratings | ~100 M | Large-scale; needs efficient data loading |
| Unique users | ~480 K | Many cold-start scenarios expected |
| Unique movies | ~17.7 K | Manageable item catalogue |
| Sparsity | ~98.82% | Extremely sparse; CF is essential |
| Rating mean | ~3.60 stars | Slight positive bias |
| Most common rating | 4 stars | Right-skewed toward higher ratings |
| Top-10% users | ~60% of all ratings | Power users dominate |
| Top-100 movies | ~12% of all ratings | Popularity bias to address |
| Date range | Oct 1998 to Dec 2005 | Temporal drift in preferences possible |

**Recommendations for modelling:**
1. Use bias terms (b_u, b_i) to capture per-user and per-movie rating offsets.
2. Apply rating normalisation/mean-centering before feeding to neural models.
3. Consider popularity-debiased metrics to avoid just recommending blockbusters.
4. Implement temporal cross-validation to prevent data leakage.

In [ ]:
import json

eda_stats = {
    'n_ratings':   int(n_ratings),
    'n_users':     int(n_users),
    'n_movies':    int(n_movies),
    'sparsity':    float(sparsity),
    'rating_mean': float(df['rating'].mean()),
    'rating_std':  float(df['rating'].std()),
}

stats_path = PROC_DIR / 'eda_stats.json'
stats_path.parent.mkdir(parents=True, exist_ok=True)
with open(stats_path, 'w') as f:
    json.dump(eda_stats, f, indent=2)

print('EDA stats saved to:', stats_path)
print(json.dumps(eda_stats, indent=2))